# Week 46, Day 3: Limits and flexibility: capacity and mixed selectivity

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w046/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Mixture fits to slot and resource data

Write em(err): start from g = 0.3 and sigma = 0.3; repeat 200 times: compute each error's probability under the normal (weighted by 1 - g) and the uniform (density g / 2 pi), set w to the normal's share, then g = 1 - mean(w) and sigma = sqrt(sum(w err^2) / sum(w)).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2008)
wrap = lambda x: np.angle(np.exp(1j * x))
sd1 = np.deg2rad(15)

def slots(N, n=1000, K=3):
    stored = r.random(n) < min(1, K / N)
    return np.where(stored, sd1 * r.normal(size=n), r.uniform(-np.pi, np.pi, n))

def resource(N, n=1000):
    return wrap(sd1 * np.sqrt(N) * r.normal(size=n))       # precision 1/sd^2 shared among N items

def em(err, iters=200):
    g, sd = 0.3, 0.3
    for _ in range(iters):
        ...
    return g, sd

sizes = [1, 2, 4, 6]
fits = {"slots": {N: em(slots(N)) for N in sizes}, "resource": {N: em(resource(N)) for N in sizes}}
for model in fits:
    print(f"{model:8s}", "; ".join(f"{N} items: guess {g:.2f}, sd {np.rad2deg(s_):.1f} deg" for N, (g, s_) in fits[model].items()))

In [ ]:
import numpy as np
def test_slots():
    "slot data: guess rate near zero for 1 and 2 items and near 1 - 3/N for 4 and 6"
    assert fits["slots"][1][0] < 0.05 and abs(fits["slots"][6][0] - 0.5) < 0.08 and abs(fits["slots"][4][0] - 0.25) < 0.08
def test_slots_sd():
    "slot data: precision roughly constant across set sizes"
    sds = [fits["slots"][N][1] for N in sizes]
    assert max(sds) - min(sds) < 0.06
def test_resource():
    "resource data: no guessing, but the error spread grows with set size"
    assert all(fits["resource"][N][0] < 0.05 for N in sizes) and fits["resource"][6][1] > 2 * fits["resource"][1][1]

test_slots()
test_slots_sd()
test_resource()
print("All checks passed")

### Exercise: Pure against mixed selectivity

Build 50 neurons of each kind for the 8 conditions: pure (a weighted sum of the three variables) and mixed (the same weighted sum, doubled, minus a random threshold, rectified). Compute the rank of the mean-subtracted response matrix and the fraction of 200 random dichotomies a perceptron can separate.

In [ ]:
import itertools
conds = np.array(list(itertools.product([0, 1], [0, 1], [0, 1])))   # 8 conditions x 3 variables

def pure(n_neurons):
    W = r.normal(size=(n_neurons, 3))
    return ...

def mixed(n_neurons):
    W = r.normal(size=(n_neurons, 3)); b = r.normal(size=n_neurons)
    return ...

def separable_fraction(R, trials=200):
    X = np.c_[R, np.ones(len(R))]; ok = 0
    for _ in range(trials):
        y = r.choice([-1, 1], 8)
        if abs(y.sum()) == 8:
            y[0] *= -1
        w = np.linalg.lstsq(X, y, rcond=None)[0]
        for _ in range(500):                                     # perceptron, starting from least squares
            wrong = np.sign(X @ w) != y
            if not wrong.any():
                break
            w += 0.1 * (y[wrong][:, None] * X[wrong]).sum(0)
        ok += not (np.sign(X @ w) != y).any()
    return ok / trials

rank, sep = {}, {}
for name, fn in [("pure", pure), ("mixed", mixed)]:
    R = fn(50)
    rank[name] = np.linalg.matrix_rank(R - R.mean(0)); sep[name] = separable_fraction(R)
    print(f"{name:5s}: dimensionality {rank[name]}, separable dichotomies {sep[name]:.2f}")

In [ ]:
import numpy as np
def test_rank():
    "pure selectivity spans 3 dimensions across 8 conditions; mixed selectivity spans 7"
    assert rank["pure"] == 3 and rank["mixed"] == 7
def test_sep():
    "mixed selectivity makes every sampled dichotomy linearly separable, pure selectivity under half"
    assert sep["mixed"] > 0.98 and sep["pure"] < 0.5

test_rank()
test_sep()
print("All checks passed")